# Clean GENA_LM Expression Inference

This notebook runs expression prediction for valid genes and 14 cell types. It is intentionally simple: configure paths, load model, prepare inputs, run one gene at a time, save a gene x cell-type CSV.

In [2]:
# 1. Paths and small test settings

from pathlib import Path

TASK_ROOT = Path("/mnt/newdata/dpanc/benchmarking/GENA_LM")
GENA_HOME = TASK_ROOT / "GENA_LM_expression_branch"

EXPERIMENT_CONFIG = GENA_HOME / "downstream_tasks/expression_prediction/inference_example/inference.yaml"
CHECKPOINT_PATH = GENA_HOME / "models/full_model/pytorch_model.bin"

INFERENCE_DIR = TASK_ROOT / "notebook_inference_valid14_clean"
JSON_DIR = TASK_ROOT / "notebook_inference_valid812/json_14"

FORWARD_INTERVALS_PATH = Path("/home/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/downstream_tasks/expression_prediction/inference_example/human.valid.forward.subsample.csv")
REVERSE_INTERVALS_PATH = None #Path("/mnt/newdata/dpanc/benchmarking/data/human.valid.reverse.csv")
GENOME_PATH = Path("/mnt/newdata/dpanc/benchmarking/data/hg38.fna")

# Keep this as 2 for smoke test. After it works, set TEST_N_GENES = None for all valid genes.
TEST_N_GENES = None

# FlashAttention model. If FlashAttention crashes, change this to True to use SDPA model.
USE_SDPA_MODEL = False

NUM_BEFORE = 512
TOKEN_LEN_FOR_FETCH = 15
PREDICTION_MATRIX_CSV = "valid14_clean_predictions.csv"

print("GENA_HOME:", GENA_HOME)
print("JSON_DIR:", JSON_DIR)
print("TEST_N_GENES:", TEST_N_GENES)


GENA_HOME: /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch
JSON_DIR: /mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid812/json_14
TEST_N_GENES: None


In [3]:
# 2. Imports and environment

import os
import sys
import json

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

os.environ["GENALM_HOME"] = str(GENA_HOME)
os.environ["TMPDIR"] = str(TASK_ROOT / "cache/tmp")
os.environ["TRITON_CACHE_DIR"] = str(TASK_ROOT / "cache/triton")
os.environ["TORCHINDUCTOR_CACHE_DIR"] = str(TASK_ROOT / "cache/torchinductor")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for p in [INFERENCE_DIR, Path(os.environ["TMPDIR"]), Path(os.environ["TRITON_CACHE_DIR"]), Path(os.environ["TORCHINDUCTOR_CACHE_DIR"] )]:
    p.mkdir(parents=True, exist_ok=True)

# If torch.compile/triton fails, fall back to eager instead of killing the notebook.
import torch._dynamo
torch._dynamo.config.suppress_errors = True

sys.path.insert(0, str(GENA_HOME))

if USE_SDPA_MODEL:
    from downstream_tasks.expression_prediction.expression_model_final_sdpa import ExpressionCounts
else:
    from downstream_tasks.expression_prediction.expression_model_final import ExpressionCounts

from downstream_tasks.expression_prediction.inference_example.inference_input_utils import prepare_inference_inputs_from_intervals

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("model implementation:", "SDPA" if USE_SDPA_MODEL else "FlashAttention")


/home/dpanc/benchmarking/GENA_LM/envs/expression_flash/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.4.0
cuda available: True
gpu: NVIDIA A100 80GB PCIe
model implementation: FlashAttention


In [4]:
# 3. Load config, model, and checkpoint

experiment_config_path = Path(EXPERIMENT_CONFIG).expanduser().absolute()

with initialize_config_dir(str(experiment_config_path.parent), version_base=None):
    experiment_config = compose(config_name=experiment_config_path.name)

model_kwargs = instantiate(experiment_config["model_kwargs"])
model = ExpressionCounts(**model_kwargs)

state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print("Loaded checkpoint:", CHECKPOINT_PATH)


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


Using ModernGENA from /mnt/newdata/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/models/modernbert_large/
missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}
qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - la

In [5]:
# 4. Load tokenizers

dna_tokenizer_name = experiment_config["args_params"]["gen_tokenizer"]
text_tokenizer_name = experiment_config["shared_dataset_params"]["text_tokenizer"]

dna_tokenizer = AutoTokenizer.from_pretrained(dna_tokenizer_name)
text_tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name, padding_side="left")

dna_max_seq_len = int(experiment_config["args_params"]["input_seq_len"])
text_max_seq_len = int(experiment_config["shared_dataset_params"]["text_max_seq_len"])

text_pad_id = text_tokenizer.pad_token_id
if text_pad_id is None:
    text_pad_id = text_tokenizer.eos_token_id if text_tokenizer.eos_token_id is not None else 0

print("DNA tokenizer:", dna_tokenizer_name)
print("Text tokenizer:", text_tokenizer_name)
print("dna_max_seq_len:", dna_max_seq_len)
print("text_max_seq_len:", text_max_seq_len)
print("text_pad_id:", text_pad_id)


DNA tokenizer: AIRI-Institute/gena-lm-bert-base-t2t
Text tokenizer: Qwen/Qwen3-Embedding-0.6B
dna_max_seq_len: 1024
text_max_seq_len: 510
text_pad_id: 151643


In [6]:
# 5. Prepare interval-based inference inputs

def make_small_interval_files(forward_path, reverse_path, n_genes, out_dir):
    if n_genes is None:
        return forward_path, reverse_path

    forward_df = pd.read_csv(forward_path, sep="\t")
    reverse_df = pd.read_csv(reverse_path, sep="\t") if reverse_path is not None else pd.DataFrame()

    n_forward = min(n_genes, len(forward_df))
    n_reverse = max(0, n_genes - n_forward)

    small_forward = forward_df.head(n_forward)
    small_reverse = reverse_df.head(n_reverse)

    small_forward_path = out_dir / f"smoke_{n_genes}.forward.csv"
    small_reverse_path = out_dir / f"smoke_{n_genes}.reverse.csv"

    small_forward.to_csv(small_forward_path, sep="\t", index=False)
    if len(small_reverse) > 0:
        small_reverse.to_csv(small_reverse_path, sep="\t", index=False)
        return small_forward_path, small_reverse_path

    return small_forward_path, None

run_forward_path, run_reverse_path = make_small_interval_files(
    FORWARD_INTERVALS_PATH,
    REVERSE_INTERVALS_PATH,
    TEST_N_GENES,
    INFERENCE_DIR,
)

print("Forward intervals used:", run_forward_path)
print("Reverse intervals used:", run_reverse_path)

prepared = prepare_inference_inputs_from_intervals(
    json_dir=JSON_DIR,
    forward_intervals_path=run_forward_path,
    reverse_intervals_path=run_reverse_path,
    genome_path=GENOME_PATH,
    gen_tokenizer=dna_tokenizer,
    text_tokenizer=text_tokenizer,
    gen_max_seq_len=dna_max_seq_len,
    text_max_seq_len=text_max_seq_len,
    cache_dir=INFERENCE_DIR,
    num_before=NUM_BEFORE,
    token_len_for_fetch=TOKEN_LEN_FOR_FETCH,
)

genes = prepared["genes"]
experiments = prepared["experiments"]
tokenized_DNA = prepared["tokenized_DNA"]
tokenized_descriptions = prepared["tokenized_descriptions"]

gene_names = list(genes.keys())
cell_type_names = list(experiments.keys())
gene_to_idx = {gene: idx for idx, gene in enumerate(gene_names)}

print("Genes used in this run:", len(gene_names))
print("Cell types:", len(cell_type_names))
print("First genes:", gene_names[:5])
print("Cell type names:", cell_type_names)
print("DNA input_ids shape:", tokenized_DNA["input_ids"].shape)


Forward intervals used: /home/dpanc/benchmarking/GENA_LM/GENA_LM_expression_branch/downstream_tasks/expression_prediction/inference_example/human.valid.forward.subsample.csv
Reverse intervals used: None


Genes used in this run: 3
Cell types: 14
First genes: ['AC010967.2', 'CHAC2', 'ERLEC1']
Cell type names: ['ENCFF035CWS', 'ENCFF083EOC', 'ENCFF123KIW', 'ENCFF236XOK', 'ENCFF242BWW', 'ENCFF329ENM', 'ENCFF361XCF', 'ENCFF494KRC', 'ENCFF602HCV', 'ENCFF660EXG', 'ENCFF664WLU', 'ENCFF761SPP', 'ENCFF784MDF', 'ENCFF857JQM']
DNA input_ids shape: torch.Size([3, 1024])


In [7]:
# 6. Run prediction: one gene x all 14 cell types

def pad_1d(x, target_len, value):
    if x.shape[0] > target_len:
        return x[:target_len]
    if x.shape[0] < target_len:
        return torch.nn.functional.pad(x, (0, target_len - x.shape[0]), value=value)
    return x

def prediction_vector_from_logits(logits, n_cells):
    logits = logits.detach().cpu().float()
    print("first logits shape:", tuple(logits.shape)) if len(prediction_matrix_data) == 0 else None

    if logits.shape[0] == n_cells:
        return logits[:, 0, 0].numpy()
    if logits.shape[0] == 1 and logits.shape[-1] == n_cells:
        return logits[0, 0, :].numpy()

    raise RuntimeError(f"Unexpected logits shape {tuple(logits.shape)} for {n_cells} cell types")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

input_ids_all = tokenized_DNA["input_ids"]
attention_mask_all = tokenized_DNA["attention_mask"]

prediction_matrix_data = []

for gene_number, gene_name in enumerate(gene_names, start=1):
    gene_idx = gene_to_idx[gene_name]

    # The model expects B*N DNA rows. For one gene and N cell types,
    # repeat the same DNA N times and set dataset_flag=True to tell the model
    # that DNA is repeated while descriptions are different.
    input_ids = input_ids_all[gene_idx:gene_idx + 1].repeat(len(cell_type_names), 1).to(device)
    attention_mask = attention_mask_all[gene_idx:gene_idx + 1].repeat(len(cell_type_names), 1).to(device)

    desc_ids_list = [tokenized_descriptions[cell]["input_ids"][gene_idx] for cell in cell_type_names]
    desc_mask_list = [tokenized_descriptions[cell]["attention_mask"][gene_idx] for cell in cell_type_names]
    max_desc_len = min(text_max_seq_len, max(x.shape[0] for x in desc_ids_list))

    desc_input_ids = torch.stack(
        [pad_1d(x, max_desc_len, text_pad_id) for x in desc_ids_list],
        dim=0,
    ).unsqueeze(0).to(device)

    desc_attention_mask = torch.stack(
        [pad_1d(x, max_desc_len, 0) for x in desc_mask_list],
        dim=0,
    ).unsqueeze(0).to(device)

    dataset_flag = torch.ones(size=(1, len(cell_type_names)), device=device, dtype=torch.bool)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16), torch.no_grad():
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            desc_input_ids=desc_input_ids,
            desc_attention_mask=desc_attention_mask,
            dataset_flag=dataset_flag,
        )

    prediction_matrix_data.append(prediction_vector_from_logits(output["logits"], len(cell_type_names)))

    if gene_number <= 5 or gene_number % 100 == 0:
        print(f"Processed {gene_number}/{len(gene_names)}: {gene_name}")

print("Done")


first logits shape: (14, 1024, 1)
Processed 1/3: AC010967.2
Processed 2/3: CHAC2
Processed 3/3: ERLEC1
Done


In [ ]:
# 7. Build and save gene x cell-type prediction table

expression_matrix = pd.DataFrame(
    np.vstack(prediction_matrix_data),
    index=gene_names,
    columns=cell_type_names,
)

predictions_df = (
    expression_matrix
    .reset_index(names="Gene")
    .melt(id_vars="Gene", var_name="Cell Type", value_name="Predicted Expression")
)

out_csv = INFERENCE_DIR / PREDICTION_MATRIX_CSV
expression_matrix.to_csv(out_csv)

print("expression_matrix shape:", expression_matrix.shape)
print("saved:", out_csv)
display(expression_matrix.head())
display(predictions_df.head())


expression_matrix shape: (3, 14)
saved: /mnt/newdata/dpanc/benchmarking/GENA_LM/notebook_inference_valid14_clean/valid14_clean_predictions.csv


,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
AC010967.2,0.322266,0.00528,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266,0.322266
CHAC2,1.164062,0.75000,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062,1.164062
ERLEC1,2.921875,2.56250,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875,2.921875


,Gene,Cell Type,Predicted Expression
0,AC010967.2,ENCFF035CWS,0.322266
1,CHAC2,ENCFF035CWS,1.164062
2,ERLEC1,ENCFF035CWS,2.921875
3,AC010967.2,ENCFF083EOC,0.005280
4,CHAC2,ENCFF083EOC,0.750000


: 